## Executive Summary

**Purpose:** Analyse a single-phase, core-type transformer from its geometry and materials.

**What it does:** Computes the peak flux density, the no-load current waveform (with optional hysteresis and dynamic loop widening), core loss, the magnetising branch, winding resistances, leakage inductance and the load performance (voltage regulation, losses, efficiency).

### Why It Matters
The no-load current of a transformer is set by the non-linear core. With an analytical chain from B(t) to i₀(t), the open-circuit characteristic, the magnetising current harmonics and the efficiency follow from datasheet-level data without FEA.

### Quick Start
**Inputs:** `TransformerCore` (limb, stack, window, stacking factor, joint gap), two `TransformerWinding`s, `CoreMaterial` (Takács loop and Bertotti coefficients), `Supply` (V₁, f).

**Outputs:** dictionaries of currents, waveforms, losses and circuit parameters.

### Scope and assumptions
- Single-phase, core-type, two concentric windings, sinusoidal supply, steady state (no inrush).
- Uniform flux density along one mean path; limbs and yokes have the same width.
- Winding resistance at DC (no skin or proximity effect); leakage flux linear.
- Load analysis uses the fundamental of the magnetising current.

## How It Fits Into the Motor Design Workflow

**Upstream:** `emachines.magnetics` — `TakacsModel` and `DynamicLoopModel` (hysteresis), `bertotti` (core loss).

**Downstream:** design decisions; a future transformer design (sizing) module.

In [ ]:
#| default_exp transformer.single_phase
#| export
from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from emachines.magnetics.hysteresis import TakacsModel, DynamicLoopModel
from emachines.magnetics.iron_loss import bertotti
from emachines.magnetics.permeability import MU0

In [ ]:
#| hide
import matplotlib.pyplot as plt
from dataclasses import replace

## Inputs

The core is described by its limb width $a$, stack depth $d$, window width $w$ and height $h$ and stacking factor $k_s$:

$$A_{fe} = k_s\,a\,d, \qquad l_c = 2(w + a) + 2(h + a), \qquad V_{fe} = A_{fe}\,l_c$$

$l_c$ is the mean flux path round one window, assuming limbs and yokes of equal width $a$. `joint_gap_total` is the total equivalent air gap of the joints in that path; it depends on how the core is built (butt, mitred, step-lap) and is not on a steel datasheet, so it defaults to 0.

`CoreMaterial` holds the static Takács loop and the Bertotti coefficients of the same steel; the dynamic loop is built from them (`DynamicLoopModel.from_bertotti`).

In [ ]:
#| export
@dataclass(frozen=True)
class TransformerCore:
    """Core geometry of a single-phase core-type transformer (SI units)."""
    limb_width: float  # [m] a, also the yoke width
    stack_depth: float  # [m] d
    window_width: float  # [m] w
    window_height: float  # [m] h
    stacking_factor: float = 0.96  # [-] k_s, iron fraction of the stack
    joint_gap_total: float = 0.0  # [m] total equivalent air gap of the joints in the flux path

    @property
    def area(self) -> float:
        """Iron cross-section A_fe = k_s a d [m²]."""
        return self.stacking_factor * self.limb_width * self.stack_depth

    @property
    def path_length(self) -> float:
        """Mean magnetic path l_c = 2(w + a) + 2(h + a) [m]."""
        a = self.limb_width
        return 2 * (self.window_width + a) + 2 * (self.window_height + a)

    @property
    def volume(self) -> float:
        """Iron volume A_fe l_c [m³]."""
        return self.area * self.path_length


@dataclass(frozen=True)
class TransformerWinding:
    """One winding (SI units)."""
    turns: int
    conductor_area: float  # [m²] copper cross-section of one turn
    mean_turn_length: float  # [m]
    radial_build: float  # [m] radial thickness of the winding


@dataclass(frozen=True)
class CoreMaterial:
    """Core steel: static Takács loop plus Bertotti loss coefficients of the same steel."""
    static: TakacsModel
    k_h: float  # Bertotti hysteresis coefficient, p_h = k_h f B̂² [W/kg]
    k_e: float  # Bertotti eddy coefficient, p_e = k_e (f B̂)² [W/kg]
    k_a: float  # Bertotti excess coefficient, p_a = k_a (f B̂)^1.5 [W/kg]
    density: float  # [kg/m³]

    @property
    def loop(self) -> DynamicLoopModel:
        """Dynamic loop of this steel (Takács + Bertotti eddy and excess fields)."""
        return DynamicLoopModel.from_bertotti(self.static, self.k_e, self.k_a, self.density)

    def loss_density(self, f: float, B_peak: float) -> float:
        """Core loss [W/kg] at frequency f [Hz] and peak flux density B_peak [T] (Bertotti)."""
        return float(bertotti(f, B_peak, self.k_h, self.k_e, self.k_a)["total"])


@dataclass(frozen=True)
class Supply:
    """Sinusoidal supply."""
    voltage: float  # [V] rms
    frequency: float  # [Hz]

In [ ]:
# Example transformer used below: about 1 kVA, 230 V / 115 V, 50 Hz.
# Illustrative dimensions and material values only — not a real or validated design.
core = TransformerCore(limb_width=40e-3, stack_depth=50e-3, window_width=40e-3, window_height=100e-3)
primary = TransformerWinding(turns=385, conductor_area=1.45e-6, mean_turn_length=0.22, radial_build=8e-3)
secondary = TransformerWinding(turns=193, conductor_area=2.9e-6, mean_turn_length=0.28, radial_build=8e-3)
supply = Supply(voltage=230.0, frequency=50.0)
steel = TakacsModel(B_s=1.6, h0=50.0, a0=0.8)        # illustrative Takács parameters
rho = 7650.0
# k_h chosen so that the Bertotti hysteresis term equals the Takács loop energy at 1.4 T;
# in practice both come from fitting the same datasheet (fit_takacs, fit_bertotti).
k_h = steel.loop_energy(steel.tip_H(1.4)) / (rho * 1.4 ** 2)
material = CoreMaterial(static=steel, k_h=k_h, k_e=5e-5, k_a=5e-4, density=rho)

assert np.isclose(core.area, 0.96 * 0.04 * 0.05)
assert np.isclose(core.path_length, 2 * 0.08 + 2 * 0.14)
print(f"A_fe = {core.area * 1e4:.2f} cm², l_c = {core.path_length * 100:.1f} cm, "
      f"iron mass = {core.volume * rho:.2f} kg, k_h = {k_h:.4f} W/(kg·Hz·T²)")

## Step 1: Peak Flux Density from the Supply Voltage

The primary equation is $v_1 = R_1 i_1 + L_{\sigma 1}\,di_1/dt + N_1 A_{fe}\,dB/dt$. At no load the drop across $R_1$ and $L_{\sigma1}$ is about the product of two small per-unit values (for example 2 % resistive drop at rated current × 5 % no-load current ≈ 0.1 % of $V_1$), so it is neglected. For $v_1 = \sqrt2 V_1 \sin\omega t$ in steady state (no DC offset):

$$B(t) = -\hat B\cos\omega t, \qquad \hat B = \frac{\sqrt2\,V_1}{\omega N_1 A_{fe}}$$

Under load the voltage across the core is $E_1$ rather than $V_1$; `load_performance` handles that.

In [ ]:
#| export
def peak_flux_density(core: TransformerCore, primary: TransformerWinding, voltage: float, frequency: float) -> float:
    """Peak core flux density B̂ = √2 V / (ω N₁ A_fe) [T] for an rms voltage across the core."""
    if voltage <= 0 or frequency <= 0:
        raise ValueError(f"voltage and frequency must be positive, got {voltage}, {frequency}")
    return float(np.sqrt(2) * voltage / (2 * np.pi * frequency * primary.turns * core.area))

In [ ]:
# Examples
B_hat = peak_flux_density(core, primary, supply.voltage, supply.frequency)
assert np.isclose(B_hat, np.sqrt(2) * 230 / (2 * np.pi * 50 * 385 * core.area))
print(f"B̂ = {B_hat:.3f} T")

## Steps 2–4: No-Load Current Waveform

The field in the core comes from the hysteresis module, $H(t)$ = `material.loop.field_waveform(B(t), dt, hysteresis, dynamic)`, and Ampère's law round the mean path, including the joints, gives

$$N_1\,i_0(t) = H(t)\,l_c + \frac{B(t)\,g_j}{\mu_0}$$

From $i_0(t)$:
- rms and peak value, crest factor and the odd harmonics (half-wave symmetry removes the even ones);
- the fundamental split against $v_1$: the in-phase part $I_{c1}$ carries the loop loss, the quadrature part $I_{m1}$ magnetises.

With a sinusoidal voltage only the fundamental carries power, so the mean of $v_1 i_0$ equals $V_1 I_{c1}$ and equals the loop energy $f\,V_{fe}\oint H\,dB$. The switches `hysteresis` and `dynamic` change the waveform only; core loss is always taken from the Bertotti formula (next step).

In [ ]:
#| export
def no_load_current(core: TransformerCore, primary: TransformerWinding, material: CoreMaterial,
                    supply: Supply, hysteresis: bool = True, dynamic: bool = True,
                    n_samples: int = 2000, n_harmonics: int = 15) -> dict:
    """No-load current waveform and its components, for sinusoidal supply voltage.

    Returns a dict with the waveforms ('t', 'v', 'B', 'H', 'i0'), 'B_peak', 'I0_rms',
    'I0_peak', 'crest_factor', 'I_c1' and 'I_m1' (rms in-phase and quadrature parts of
    the fundamental), 'harmonics_rms' (orders 1..n_harmonics) and 'P_loop' [W], the
    power drawn by the loop (mean of v·i0).
    """
    f, V = supply.frequency, supply.voltage
    B_hat = peak_flux_density(core, primary, V, f)
    t = np.arange(n_samples) / (n_samples * f)
    wt = 2 * np.pi * f * t
    v = np.sqrt(2) * V * np.sin(wt)
    B = -B_hat * np.cos(wt)
    H = material.loop.field_waveform(B, t[1], hysteresis=hysteresis, dynamic=dynamic)
    i0 = (H * core.path_length + B * core.joint_gap_total / MU0) / primary.turns
    c = np.fft.rfft(i0) / n_samples * 2                     # peak amplitudes; c[1] = fundamental
    a1 = np.mean(i0 * np.sin(wt)) * 2                      # in phase with v
    b1 = np.mean(i0 * np.cos(wt)) * 2                      # quadrature (magnetising lags v by 90°)
    I0_rms = float(np.sqrt(np.mean(i0 ** 2)))
    return {"t": t, "v": v, "B": B, "H": H, "i0": i0, "B_peak": B_hat,
            "I0_rms": I0_rms, "I0_peak": float(np.max(np.abs(i0))),
            "crest_factor": float(np.max(np.abs(i0)) / I0_rms),
            "I_c1": float(a1 / np.sqrt(2)), "I_m1": float(abs(b1) / np.sqrt(2)),
            "harmonics_rms": np.abs(c[1:n_harmonics + 1]) / np.sqrt(2),
            "P_loop": float(np.mean(v * i0))}

In [ ]:
# Examples
nl = no_load_current(core, primary, material, supply)
# energy check: power drawn = V·I_c1 = loop energy f·V_fe·∮H dB
assert np.isclose(nl["P_loop"], supply.voltage * nl["I_c1"], rtol=1e-3)
W_cycle = material.loop.cycle_energy(nl["B"], nl["t"][1])
assert np.isclose(nl["P_loop"], supply.frequency * core.volume * W_cycle, rtol=1e-3)
# half-wave symmetry: no even harmonics
assert np.all(nl["harmonics_rms"][1::2] < 1e-9 * nl["I0_rms"])
print(f"I0 = {nl['I0_rms'] * 1e3:.1f} mA rms, crest factor {nl['crest_factor']:.2f}, "
      f"I_c1 = {nl['I_c1'] * 1e3:.1f} mA, I_m1 = {nl['I_m1'] * 1e3:.1f} mA")
print("odd harmonics, % of fundamental:",
      {k: round(float(100 * nl['harmonics_rms'][k - 1] / nl['harmonics_rms'][0]), 1) for k in (3, 5, 7, 9)})

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
deg = np.degrees(2 * np.pi * supply.frequency * nl["t"])
for (h, d), style, label in [((False, False), "k:", "plain B–H"), ((True, False), "C1--", "static loop"),
                             ((True, True), "C0", "static loop + 50 Hz widening")]:
    r = no_load_current(core, primary, material, supply, hysteresis=h, dynamic=d)
    ax.plot(deg, r["i0"] * 1e3, style, lw=1.4, label=label)
ax2 = ax.twinx(); ax2.plot(deg, nl["v"], color="0.6", lw=0.8); ax2.set_ylabel("v₁ (V)", color="0.5")
ax.set_xlabel("ωt (deg)"); ax.set_ylabel("i₀ (mA)")
ax.set_title("No-load current, illustrative transformer"); ax.grid(True, alpha=0.4); ax.legend(fontsize=8)
plt.show()

## Step 5: Core Loss and the Magnetising Branch

Core loss is always taken from the Bertotti formula, $P_{fe} = p(f, \hat B)\,\rho\,V_{fe}$, whatever the switches. With both switches on and a loop fitted to the same steel the two agree (the loop area equals the Bertotti loss per cycle); with a switch off the waveform no longer carries all of the loss, but the loss is still counted.

The magnetising branch at that voltage:

$$R_c = \frac{V_1^2}{P_{fe}}, \qquad X_m = \frac{V_1}{I_{m1}}$$

Both depend on $V_1$ because the iron is non-linear; evaluating them over a range of voltages gives the open-circuit characteristic.

In [ ]:
#| export
def core_loss(core: TransformerCore, material: CoreMaterial, frequency: float, B_peak: float) -> float:
    """Core loss [W] from the Bertotti formula over the iron volume."""
    return material.loss_density(frequency, B_peak) * material.density * core.volume


def magnetising_branch(core: TransformerCore, primary: TransformerWinding, material: CoreMaterial,
                       supply: Supply, hysteresis: bool = True, dynamic: bool = True) -> dict:
    """Shunt branch at the supply voltage: 'R_c' and 'X_m' [Ω] (primary side), 'P_fe' [W], 'I_m1' [A].

    R_c = V²/P_fe with P_fe from the Bertotti formula; X_m = V/I_m1 with I_m1 the
    quadrature fundamental of the no-load current.
    """
    nl = no_load_current(core, primary, material, supply, hysteresis, dynamic)
    P_fe = core_loss(core, material, supply.frequency, nl["B_peak"])
    return {"R_c": supply.voltage ** 2 / P_fe, "X_m": supply.voltage / nl["I_m1"],
            "P_fe": P_fe, "I_m1": nl["I_m1"], "B_peak": nl["B_peak"]}

In [ ]:
# Examples
mb = magnetising_branch(core, primary, material, supply)
P_fe = core_loss(core, material, 50.0, mb["B_peak"])
assert np.isclose(mb["R_c"], 230 ** 2 / P_fe)
# at 1.4 T k_h was matched to the loop, so the loop power equals the Bertotti loss there
V_14 = supply.voltage * 1.4 / B_hat                     # voltage giving B̂ = 1.4 T
nl14 = no_load_current(core, primary, material, Supply(V_14, 50.0))
assert np.isclose(nl14["P_loop"], core_loss(core, material, 50.0, 1.4), rtol=2e-3)
print(f"P_fe = {mb['P_fe']:.2f} W, loop power = {nl['P_loop']:.2f} W at {B_hat:.3f} T; "
      f"R_c = {mb['R_c']:.0f} Ω, X_m = {mb['X_m']:.0f} Ω")

In [ ]:
V_sweep = np.linspace(0.4, 1.1, 15) * supply.voltage
occ = [no_load_current(core, primary, material, Supply(V, 50.0)) for V in V_sweep]
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot([r["I0_rms"] * 1e3 for r in occ], V_sweep, "o-", ms=3, label="I₀ rms")
ax.plot([r["I_m1"] * 1e3 for r in occ], V_sweep, "s--", ms=3, label="I_m1 (fundamental, magnetising)")
ax.set_xlabel("current (mA)"); ax.set_ylabel("V₁ (V)")
ax.set_title("Open-circuit characteristic, illustrative transformer"); ax.grid(True, alpha=0.4); ax.legend(fontsize=8)
plt.show()

## Steps 6–7: Winding Resistance and Leakage Inductance

**Resistance** of a winding with $N$ turns of copper area $A_{cu}$ and mean turn length $l_{mt}$, at temperature $T$:

$$R = \rho_{20}\,[1 + \alpha_{20}(T - 20)]\,\frac{N\,l_{mt}}{A_{cu}}$$

with annealed copper $\rho_{20} = 1.7241\cdot10^{-8}\ \Omega$m and $\alpha_{20} = 0.00393$ /K (IEC 60028). DC resistance: skin and proximity effects are not included.

**Leakage inductance** of two concentric windings, referred to the primary (Rogowski):

$$L_\sigma = \frac{\mu_0 N_1^2\,l_{mt}}{h_w}\left(\frac{a_1}{3} + \frac{a_2}{3} + \delta\right)k_R, \qquad k_R = 1 - \frac{1 - e^{-\pi h_w/\lambda}}{\pi h_w/\lambda}, \quad \lambda = a_1 + a_2 + \delta$$

$a_1, a_2$ are the radial builds, $\delta$ the gap between the windings and $h_w$ the winding height; $l_{mt}$ is taken as the mean of the two windings' mean turn lengths. $k_R$ corrects for the fringing of the leakage flux at the winding ends.

In [ ]:
#| export
def winding_resistance(winding: TransformerWinding, temperature_C: float = 75.0,
                       resistivity_20C: float = 1.7241e-8, alpha_20C: float = 3.93e-3) -> float:
    """DC resistance [Ω] of a winding at temperature_C (annealed copper defaults, IEC 60028)."""
    rho = resistivity_20C * (1 + alpha_20C * (temperature_C - 20.0))
    return rho * winding.turns * winding.mean_turn_length / winding.conductor_area


def rogowski_factor(winding_height: float, radial_width: float) -> float:
    """Rogowski factor k_R for a leakage channel of total radial width λ = a1 + a2 + δ."""
    if winding_height <= 0 or radial_width <= 0:
        raise ValueError("winding_height and radial_width must be positive")
    x = np.pi * winding_height / radial_width
    return float(1 - (1 - np.exp(-x)) / x)


def leakage_inductance(primary: TransformerWinding, secondary: TransformerWinding,
                       gap: float, winding_height: float) -> float:
    """Total leakage inductance [H] of two concentric windings, referred to the primary."""
    if gap < 0:
        raise ValueError(f"gap must be >= 0, got {gap}")
    a1, a2 = primary.radial_build, secondary.radial_build
    l_mt = 0.5 * (primary.mean_turn_length + secondary.mean_turn_length)
    k_R = rogowski_factor(winding_height, a1 + a2 + gap)
    return MU0 * primary.turns ** 2 * l_mt / winding_height * (a1 / 3 + a2 / 3 + gap) * k_R

In [ ]:
# Examples
R1, R2 = winding_resistance(primary), winding_resistance(secondary)
assert np.isclose(R1, 1.7241e-8 * (1 + 3.93e-3 * 55) * 385 * 0.22 / 1.45e-6)
assert np.isclose(winding_resistance(primary, 20.0) / winding_resistance(primary, 75.0), 1 / (1 + 3.93e-3 * 55))
assert 0 < rogowski_factor(90e-3, 18e-3) < 1 and rogowski_factor(10.0, 18e-3) > 0.999   # → 1 for tall windings
L_s = leakage_inductance(primary, secondary, gap=2e-3, winding_height=90e-3)
print(f"R1 = {R1:.3f} Ω, R2 = {R2:.3f} Ω at 75 °C; L_σ = {L_s * 1e3:.2f} mH "
      f"(X_σ = {2 * np.pi * 50 * L_s:.2f} Ω) referred to the primary")

## Step 8: Load Performance

The load is given as an apparent power $S$ at power factor $\cos\varphi$ on the rated secondary voltage $V_{2n} = V_1 N_2/N_1$, which fixes a constant load impedance $Z_L = (V_{2n}^2/S)\,e^{\pm j\varphi}$ (+ lagging). Everything is referred to the primary with $k = N_1/N_2$:

- series impedances $Z_1 = R_1 + jX_\sigma/2$, $Z_2' = k^2 R_2 + jX_\sigma/2$ (leakage split equally between the windings), load $Z_L' = k^2 Z_L$;
- shunt admittance at the core voltage, $Y_m(E_1) = 1/R_c(E_1) - j/X_m(E_1)$.

$Y_m$ depends on $|E_1|$, so the circuit is solved by fixed-point iteration from $E_1 = V_1$:

$$E_1 = \frac{V_1}{1 + Z_1\left[Y_m(E_1) + 1/(Z_2' + Z_L')\right]}$$

Then $I_2' = E_1/(Z_2' + Z_L')$, $I_1 = E_1[Y_m + 1/(Z_2' + Z_L')]$, the core flux density is $\hat B(|E_1|)$, and

$$\eta = \frac{P_{out}}{P_{out} + P_{cu} + P_{fe}}, \qquad \text{regulation} = \frac{V_{2n} - V_2}{V_{2n}}$$

The magnetising branch uses only the fundamental of the no-load current.

In [ ]:
#| export
def load_performance(core: TransformerCore, primary: TransformerWinding, secondary: TransformerWinding,
                     material: CoreMaterial, supply: Supply, load_va: float, power_factor: float,
                     gap: float, winding_height: float, lagging: bool = True, temperature_C: float = 75.0,
                     hysteresis: bool = True, dynamic: bool = True, tol: float = 1e-9, max_iter: int = 50) -> dict:
    """Steady-state load performance from the T equivalent circuit, referred to the primary.

    The load is load_va [VA] at power_factor on the rated secondary voltage V1·N2/N1
    (a constant impedance). The magnetising branch is evaluated at the core voltage E1
    by fixed-point iteration. Returns currents, voltages, losses, efficiency and regulation.
    """
    if load_va <= 0 or not 0 < power_factor <= 1:
        raise ValueError(f"load_va must be > 0 and power_factor in (0, 1], got {load_va}, {power_factor}")
    k = primary.turns / secondary.turns
    V2n = supply.voltage / k
    phi = np.arccos(power_factor) * (1 if lagging else -1)
    Z_L = V2n ** 2 / load_va * np.exp(1j * phi)
    X_s = 2 * np.pi * supply.frequency * leakage_inductance(primary, secondary, gap, winding_height)
    R1, R2 = winding_resistance(primary, temperature_C), winding_resistance(secondary, temperature_C)
    Z1, Z2p, ZLp = R1 + 0.5j * X_s, k ** 2 * R2 + 0.5j * X_s, k ** 2 * Z_L

    E1 = complex(supply.voltage)
    for _ in range(max_iter):
        mb = magnetising_branch(core, primary, material, Supply(abs(E1), supply.frequency), hysteresis, dynamic)
        Y_m = 1 / mb["R_c"] - 1j / mb["X_m"]
        E1_new = supply.voltage / (1 + Z1 * (Y_m + 1 / (Z2p + ZLp)))
        if abs(E1_new - E1) < tol * supply.voltage:
            E1 = E1_new
            break
        E1 = E1_new
    else:
        raise RuntimeError("load_performance: E1 iteration did not converge")
    mb = magnetising_branch(core, primary, material, Supply(abs(E1), supply.frequency), hysteresis, dynamic)
    Y_m = 1 / mb["R_c"] - 1j / mb["X_m"]
    I2p = E1 / (Z2p + ZLp)
    I1 = E1 * Y_m + I2p
    V2 = abs(I2p * ZLp) / k
    P_out = abs(I2p) ** 2 * ZLp.real
    P_cu = abs(I1) ** 2 * R1 + abs(I2p) ** 2 * Z2p.real
    P_fe = mb["P_fe"]
    return {"I1": abs(I1), "I2": abs(I2p) * k, "V2": V2, "E1": abs(E1), "B_peak": mb["B_peak"],
            "P_out": P_out, "P_cu": P_cu, "P_fe": P_fe, "P_in": float((supply.voltage * np.conj(I1)).real),
            "efficiency": P_out / (P_out + P_cu + P_fe), "regulation": (V2n - V2) / V2n}

In [ ]:
# Examples
lp = load_performance(core, primary, secondary, material, supply, load_va=1000.0, power_factor=0.8,
                      gap=2e-3, winding_height=90e-3)
assert np.isclose(lp["P_in"], lp["P_out"] + lp["P_cu"] + lp["P_fe"], rtol=1e-9)   # power balance
assert lp["E1"] < supply.voltage and lp["B_peak"] < B_hat                          # core sees E1 < V1
print(f"1 kVA, pf 0.8 lagging: V2 = {lp['V2']:.1f} V (regulation {100 * lp['regulation']:.2f} %), "
      f"I1 = {lp['I1']:.2f} A, P_cu = {lp['P_cu']:.1f} W, P_fe = {lp['P_fe']:.2f} W, η = {100 * lp['efficiency']:.2f} %, "
      f"B̂ = {lp['B_peak']:.3f} T")

## Tests

In [ ]:
# switches: plain B–H draws no loop power; static loop draws less than the dynamic loop
r_off = no_load_current(core, primary, material, supply, hysteresis=False, dynamic=False)
r_static = no_load_current(core, primary, material, supply, dynamic=False)
assert abs(r_off["P_loop"]) < 1e-9 and abs(r_off["I_c1"]) < 1e-9
assert 0 < r_static["P_loop"] < nl["P_loop"]
# core loss does not depend on the switches
assert magnetising_branch(core, primary, material, supply, False, False)["P_fe"] == mb["P_fe"]

# joints: add B g_j / (μ0 N1) to the current, raising the magnetising current
core_j = replace(core, joint_gap_total=80e-6)
nl_j = no_load_current(core_j, primary, material, supply)
assert np.allclose(nl_j["i0"] - nl["i0"], nl["B"] * 80e-6 / MU0 / primary.turns)
assert nl_j["I_m1"] > nl["I_m1"] and np.isclose(nl_j["P_loop"], nl["P_loop"], rtol=1e-6)

# B̂ scales as V / (f N)
assert np.isclose(peak_flux_density(core, primary, 115, 50), B_hat / 2)
assert np.isclose(peak_flux_density(core, primary, 230, 60), B_hat * 50 / 60)

# load: unity pf and a lighter load give smaller regulation; efficiency below 1
lp_light = load_performance(core, primary, secondary, material, supply, 250.0, 1.0, 2e-3, 90e-3)
assert lp_light["regulation"] < lp["regulation"] and 0 < lp_light["efficiency"] < 1
assert np.isclose(lp_light["P_in"], lp_light["P_out"] + lp_light["P_cu"] + lp_light["P_fe"], rtol=1e-9)

for bad in [lambda: peak_flux_density(core, primary, 0, 50), lambda: rogowski_factor(0, 1e-2),
            lambda: leakage_inductance(primary, secondary, -1e-3, 0.09),
            lambda: load_performance(core, primary, secondary, material, supply, 1000, 1.2, 2e-3, 0.09)]:
    try:
        bad()
    except ValueError:
        pass
    else:
        raise AssertionError("no ValueError")
print("✓ Transformer tests passed")

## References

- McLyman, C.W.T. (2004). *Transformer and Inductor Design Handbook* (3rd ed.). Marcel Dekker.
- Kulkarni, S.V., Khaparde, S.A. (2004). *Transformer Engineering: Design and Practice*. Marcel Dekker — leakage reactance, Rogowski factor.
- Say, M.G. (1976). *Alternating Current Machines*. Pitman — equivalent circuit, regulation, efficiency.
- IEC 60028 — International standard of resistance for copper (ρ₂₀, α₂₀).